# Model: Biased SAT Sensor (Experimental Dataset, Cross-Season Evaluation)

## Feature choice, informed by notebook 10's EDA

Per notebook 10: RTU_TOT_WATT was the most robust cross-season finding (increases
meaningfully and directionally with bias magnitude, checked in both Winter_2022 and
Spring_2021). RTU_SA_TEMP is mostly invisible to this fault (control loop chases the
biased reading) EXCEPT one real exception found in Spring at the most extreme
severity - included anyway to test whether the model can usefully exploit that
partial signal despite its inconsistency.

## Seasons

Same design as notebook 19: train on Winter_2022 + Spring_2021 (both directly
validated in the EDA for this fault), evaluate on held-out Summer_2021 (a genuinely
fresh test, never examined for this fault before). Fall_2020 excluded, consistent
with treating it as a structurally atypical season throughout this dataset's work.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from src.features.build_experimental_features import build_experimental_feature_table  # noqa: E402

feature_cols = ("RTU_TOT_WATT", "RTU_SA_TEMP")

train_table = pd.concat([
    build_experimental_feature_table(
        baseline_path=f"../data/raw/experimental/ERTU_{season}.csv",
        fault_paths={
            f"sat_neg4_{season}": f"../data/raw/experimental/SA_temp_bias_-4_{season}.csv",
            f"sat_neg2_{season}": f"../data/raw/experimental/SA_temp_bias_-2_{season}.csv",
            f"sat_pos2_{season}": f"../data/raw/experimental/SA_temp_bias_2_{season}.csv",
            f"sat_pos4_{season}": f"../data/raw/experimental/SA_temp_bias_4_{season}.csv",
        },
        feature_cols=feature_cols,
    )
    for season in ["Winter_2022", "Spring_2021"]
], ignore_index=True)

test_table = build_experimental_feature_table(
    baseline_path="../data/raw/experimental/ERTU_Summer_2021.csv",
    fault_paths={
        "sat_neg4_Summer_2021": "../data/raw/experimental/SA_temp_bias_-4_Summer_2021.csv",
        "sat_neg2_Summer_2021": "../data/raw/experimental/SA_temp_bias_-2_Summer_2021.csv",
        "sat_pos2_Summer_2021": "../data/raw/experimental/SA_temp_bias_2_Summer_2021.csv",
        "sat_pos4_Summer_2021": "../data/raw/experimental/SA_temp_bias_4_Summer_2021.csv",
    },
    feature_cols=feature_cols,
)

print(f"Train shape: {train_table.shape}, labels:\n{train_table['label'].value_counts()}")
print(f"\nTest shape: {test_table.shape}, labels:\n{test_table['label'].value_counts()}")

Train shape: (10800, 5), labels:
label
1    7200
0    3600
Name: count, dtype: int64

Test shape: (5400, 5), labels:
label
1    3600
0    1800
Name: count, dtype: int64


## Cross-season evaluation: train on Winter+Spring, test on held-out Summer

In [2]:
X_train = train_table[list(feature_cols)]
y_train = train_table["label"]
X_test = test_table[list(feature_cols)]
y_test = test_table["label"]

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Cross-season test: trained on Winter+Spring, evaluated on Summer ===")
print(classification_report(y_test, y_pred, target_names=["baseline", "sat_bias"]))

=== Cross-season test: trained on Winter+Spring, evaluated on Summer ===
              precision    recall  f1-score   support

    baseline       0.12      0.00      0.00      1800
    sat_bias       0.67      1.00      0.80      3600

    accuracy                           0.67      5400
   macro avg       0.40      0.50      0.40      5400
weighted avg       0.49      0.67      0.53      5400



## Result: near-total collapse (baseline recall ~0.00) — same pattern as OA damper
## stuck, likely the same underlying cause

Baseline recall ≈0.00, precision only 0.12 - the model almost never recognizes
Summer's genuine baseline as normal, mirroring OA damper stuck's raw-feature
collapse exactly. Plausible explanation, directly analogous to that case: RTU_TOT_
WATT's real value is driven heavily by cooling load, which is naturally much higher
in Summer baseline than in Winter/Spring baseline - the model likely learned
"normal = low/moderate power" from the cooler training seasons, so Summer's
genuinely normal (but higher) power draw looks exactly like the elevated draw the
fault itself produces.

This is a second, independent confirmation that RAW absolute values of
season-sensitive signals do not transfer across seasons for this dataset - not a
fluke specific to OA damper stuck's damper position, but a real, general property
of this dataset's models. Checking whether the same OA_TEMP-residualization
approach that partially helped OA damper stuck also partially helps here.

In [3]:
from sklearn.linear_model import LinearRegression

train_baseline_rows = train_table[train_table["label"] == 0]

# Need OA_TEMP for residualization - rebuild train/test tables with it included
feature_cols_with_oa = ("RTU_TOT_WATT", "RTU_SA_TEMP", "RTU_OA_TEMP")

train_table = pd.concat([
    build_experimental_feature_table(
        baseline_path=f"../data/raw/experimental/ERTU_{season}.csv",
        fault_paths={
            f"sat_neg4_{season}": f"../data/raw/experimental/SA_temp_bias_-4_{season}.csv",
            f"sat_neg2_{season}": f"../data/raw/experimental/SA_temp_bias_-2_{season}.csv",
            f"sat_pos2_{season}": f"../data/raw/experimental/SA_temp_bias_2_{season}.csv",
            f"sat_pos4_{season}": f"../data/raw/experimental/SA_temp_bias_4_{season}.csv",
        },
        feature_cols=feature_cols_with_oa,
    )
    for season in ["Winter_2022", "Spring_2021"]
], ignore_index=True)

test_table = build_experimental_feature_table(
    baseline_path="../data/raw/experimental/ERTU_Summer_2021.csv",
    fault_paths={
        "sat_neg4_Summer_2021": "../data/raw/experimental/SA_temp_bias_-4_Summer_2021.csv",
        "sat_neg2_Summer_2021": "../data/raw/experimental/SA_temp_bias_-2_Summer_2021.csv",
        "sat_pos2_Summer_2021": "../data/raw/experimental/SA_temp_bias_2_Summer_2021.csv",
        "sat_pos4_Summer_2021": "../data/raw/experimental/SA_temp_bias_4_Summer_2021.csv",
    },
    feature_cols=feature_cols_with_oa,
)

watt_weather_model = LinearRegression()
train_baseline_rows = train_table[train_table["label"] == 0]
watt_weather_model.fit(train_baseline_rows[["RTU_OA_TEMP"]], train_baseline_rows["RTU_TOT_WATT"])

train_table["watt_residual"] = train_table["RTU_TOT_WATT"] - watt_weather_model.predict(train_table[["RTU_OA_TEMP"]])
test_table["watt_residual"] = test_table["RTU_TOT_WATT"] - watt_weather_model.predict(test_table[["RTU_OA_TEMP"]])

print("Residual summary by label, TEST (Summer, held out):")
print(test_table.groupby("label")["watt_residual"].describe())

Residual summary by label, TEST (Summer, held out):
        count        mean          std          min          25%         50%  \
label                                                                          
0      1800.0 -441.038037  2139.460990 -7365.894983 -1476.381510 -917.939220   
1      3600.0  -22.295347  3056.149591 -7980.741522 -1297.238069 -584.130752   

              75%          max  
label                           
0      153.067399  6680.368999  
1      346.958728  6576.216135  


## Residual shows weak separation — testing directly, expecting limited improvement

Baseline residual mean (-441.0) and fault residual mean (-22.3) are much closer
together relative to their spread than OA damper stuck's residual was - weaker
separation. Testing the classifier directly rather than judging from summary
statistics alone, but expecting at most a modest improvement, not a clean fix.

In [4]:
model_resid = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_resid.fit(train_table[["watt_residual"]], train_table["label"])
y_pred_resid = model_resid.predict(test_table[["watt_residual"]])

print("=== Cross-season test WITH watt_residual feature ===")
print(classification_report(y_test, y_pred_resid, target_names=["baseline", "sat_bias"]))

=== Cross-season test WITH watt_residual feature ===
              precision    recall  f1-score   support

    baseline       0.34      0.05      0.09      1800
    sat_bias       0.67      0.95      0.78      3600

    accuracy                           0.65      5400
   macro avg       0.50      0.50      0.44      5400
weighted avg       0.56      0.65      0.55      5400



## Result: residualization barely helps here — a genuinely harder cross-season
## problem than OA damper stuck

| | Raw features | With watt_residual |
|---|---|---|
| Baseline recall | 0.00 | 0.05 |
| Baseline precision | 0.12 | 0.34 |

A much smaller improvement than OA damper stuck's residualization gave (0.11->0.34
there, vs. 0.00->0.05 here). This fault's cross-season generalization problem is
NOT well-addressed by the same fix that partially worked for OA damper stuck -
consistent with the weaker residual separation already observed in the summary
statistics before testing.

**Plausible reason for the difference**: OA damper stuck's core signal (damper
position) has a fairly direct, single-cause relationship with OA_TEMP (the
documented economizer control logic). RTU_TOT_WATT's relationship with OA_TEMP is
likely more complex - total power draw depends on cooling load, which itself
depends on OA_TEMP nonlinearly and also on humidity, occupancy, and other factors
not captured by a simple linear regression against OA_TEMP alone. A linear
residualization is a weaker approximation here than it was for damper position.

**Honest conclusion**: biased SAT sensor's cross-season generalization is a
genuinely harder, still largely UNSOLVED problem - worse than OA damper stuck's
partial fix, not fully addressed by anything tried in this notebook. Flagged as an
open item requiring either a more sophisticated feature (e.g. a nonlinear cooling-
load model, not attempted here) or accepting season-specific models/thresholds as
the practical path forward, same conclusion reached for OA damper stuck but with
less mitigation available so far.

## Summary: biased SAT sensor, cross-season model (Experimental dataset)

Second Experimental-dataset model built. Raw RTU_TOT_WATT/RTU_SA_TEMP features
collapse on held-out Summer (baseline recall ~0.00), same pattern as OA damper
stuck. Unlike OA damper stuck, OA_TEMP-residualization barely helps here (recall
0.00->0.05, vs. 0.11->0.34 for OA damper stuck) - likely because RTU_TOT_WATT's
relationship with weather is more complex (cooling load depends on humidity,
occupancy, and nonlinear effects) than a simple linear regression captures.

**Status**: cross-season generalization for this fault is a genuinely harder,
largely unsolved problem - worse than OA damper stuck's partial mitigation.
Flagged as an open item for future work (a nonlinear weather model, or accepting
season-specific thresholds), not resolved here, consistent with this project's
standard of documenting real limitations honestly rather than forcing a fix.